
# 02 — Merger dan Audit Lintas Anggota V2

Notebook ini memperbaiki masalah file kosong pada versi sebelumnya.

## Penyebab masalah versi lama

Kolom `totalScore` pada CSV memakai format desimal Indonesia, misalnya `5,0`.  
Versi lama membaca file tanpa parameter `decimal=","`, sehingga rating berubah menjadi `NaN`.  
Akibatnya seluruh data dianggap tidak valid dan output menjadi kosong.

## Output utama

- `02_Data_Final_Sebelum_NLP_V2.csv`
- `02_Hasil_Merger_Audit_Lintas_Anggota_V2.xlsx`

Workbook hanya mempunyai lima sheet agar mudah dipahami:

1. `Data_Final_Sebelum_NLP`
2. `Ringkasan`
3. `Rekonsiliasi`
4. `Audit_Duplikat`
5. `Audit_Anomali`


In [ ]:

# 1. IMPORT
import os
import re
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd

try:
    from google.colab import files
    RUNNING_IN_COLAB = True
except ImportError:
    RUNNING_IN_COLAB = False

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 150)
print("Library berhasil dimuat.")


In [ ]:

# 2. KONFIGURASI
INPUT_FILE = "02_Data_Reviewed_Gabungan_Sebelum_Dedup.csv"

OUTPUT_CSV = "02_Data_Final_Sebelum_NLP_V2.csv"
OUTPUT_AUDIT_CSV = "02_Hasil_Merger_Audit_Lintas_Anggota_V2.csv"

REVIEW_SEPARATOR = " ||| "
MIN_CROSS_ENTITY_TEXT_LENGTH = 40

print("Konfigurasi selesai.")


In [ ]:

# 3. UPLOAD INPUT
if not os.path.exists(INPUT_FILE):
    if not RUNNING_IN_COLAB:
        raise FileNotFoundError(INPUT_FILE)

    print("Upload:", INPUT_FILE)
    uploaded = files.upload()

    if INPUT_FILE not in uploaded:
        csv_files = [name for name in uploaded if name.lower().endswith(".csv")]
        if len(csv_files) != 1:
            raise FileNotFoundError(
                "Upload tepat satu file CSV hasil review gabungan."
            )
        os.rename(csv_files[0], INPUT_FILE)

print("File input ditemukan.")


In [ ]:

# 4. BACA DATA — FIX DESIMAL KOMA
df_raw = pd.read_csv(
    INPUT_FILE,
    sep=";",
    encoding="utf-8-sig",
    on_bad_lines="warn",
    decimal=",",  # FIX UTAMA
)

print("Ukuran input:", df_raw.shape)
display(df_raw.head(3))

if len(df_raw) == 0:
    raise ValueError("File input kosong.")


In [ ]:

# 5. FUNGSI BANTU
def clean_scalar(value):
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value).strip()

def normalize_text(value):
    return clean_scalar(value).lower()

INVALID_TEXT_VALUES = {
    "", "nan", "none", "null", "#name?", "-", "tidak ada ulasan"
}

def split_review_segments(value):
    value = clean_scalar(value)

    if normalize_text(value) in INVALID_TEXT_VALUES:
        return []

    result = []
    seen = set()

    for part in re.split(r"\s*\|\|\|\s*", value):
        original = clean_scalar(part)
        normalized = normalize_text(original)

        if normalized in INVALID_TEXT_VALUES:
            continue

        if normalized not in seen:
            seen.add(normalized)
            result.append(original)

    return result

print("Fungsi bantu siap.")


In [ ]:

# 6. VALIDASI DAN PERSIAPAN
required_columns = [
    "anggota", "entity_key", "title", "totalScore",
    "reviewsCount", "street", "city", "categoryName",
    "url", "text", "_source_file", "_run_id",
    "_crawl_date", "final_decision",
]

missing_columns = [
    column for column in required_columns
    if column not in df_raw.columns
]

if missing_columns:
    raise ValueError(f"Kolom wajib tidak tersedia: {missing_columns}")

# Fallback tambahan agar format angka tetap aman.
df_raw["totalScore"] = pd.to_numeric(
    df_raw["totalScore"]
    .astype(str)
    .str.replace(",", ".", regex=False),
    errors="coerce",
)

df_raw["reviewsCount"] = pd.to_numeric(
    df_raw["reviewsCount"],
    errors="coerce",
)

if df_raw["totalScore"].isna().all():
    raise ValueError(
        "Semua rating menjadi NaN. Periksa format desimal CSV."
    )

df_raw["_crawl_dt"] = pd.to_datetime(
    df_raw["_crawl_date"],
    errors="coerce",
)

df_raw["_segments"] = df_raw["text"].apply(
    split_review_segments
)

df_raw["_text_count"] = df_raw["_segments"].apply(len)

completeness_columns = [
    "title", "totalScore", "reviewsCount",
    "street", "city", "categoryName", "url", "text",
]

df_raw["_completeness"] = sum(
    df_raw[column].notna()
    & df_raw[column].astype(str).str.strip().ne("")
    for column in completeness_columns
)

df_raw["_has_date"] = (
    df_raw["_crawl_dt"].notna().astype(int)
)

df_raw["_has_url"] = (
    df_raw["url"].notna()
    & df_raw["url"].astype(str).str.strip().ne("")
).astype(int)

print("Jumlah input:", len(df_raw))
print("Entity unik :", df_raw["entity_key"].nunique())
print("Duplikat    :", df_raw.duplicated("entity_key").sum())


In [ ]:

# 7. AUDIT TEKS IDENTIK LINTAS ENTITAS
text_entities = defaultdict(set)

for _, row in df_raw.iterrows():
    for segment in row["_segments"]:
        text_entities[normalize_text(segment)].add(
            row["entity_key"]
        )

suspicious_texts = {
    text
    for text, entities in text_entities.items()
    if len(entities) > 1
    and len(text) >= MIN_CROSS_ENTITY_TEXT_LENGTH
}

print(
    "Teks identik panjang lintas entitas:",
    len(suspicious_texts)
)


In [ ]:

# 8. DEDUPLIKASI LINTAS ANGGOTA
final_rows = []
duplicate_records = []

for entity_key, group in df_raw.groupby(
    "entity_key",
    dropna=False,
    sort=False,
):
    group = group.sort_values(
        [
            "_has_date",
            "_crawl_dt",
            "_completeness",
            "reviewsCount",
            "_text_count",
            "_has_url",
        ],
        ascending=[
            False, False, False,
            False, False, False,
        ],
        na_position="last",
    )

    canonical = group.iloc[0].copy()

    merged_segments = []
    seen_segments = set()
    removed_cross_entity = 0

    for _, source_row in group.iterrows():
        for segment in source_row["_segments"]:
            normalized = normalize_text(segment)

            if normalized in suspicious_texts:
                removed_cross_entity += 1
                continue

            if normalized not in seen_segments:
                seen_segments.add(normalized)
                merged_segments.append(segment)

    canonical["text"] = REVIEW_SEPARATOR.join(
        merged_segments
    )
    canonical["jumlah_teks_untuk_sentimen"] = len(
        merged_segments
    )
    canonical["anggota_sumber"] = " | ".join(
        dict.fromkeys(
            clean_scalar(value)
            for value in group["anggota"]
            if clean_scalar(value)
        )
    )
    canonical["source_file_list"] = " | ".join(
        dict.fromkeys(
            clean_scalar(value)
            for value in group["_source_file"]
            if clean_scalar(value)
        )
    )
    canonical["run_id_list"] = " | ".join(
        dict.fromkeys(
            clean_scalar(value)
            for value in group["_run_id"]
            if clean_scalar(value)
        )
    )
    canonical["jumlah_baris_sumber"] = len(group)
    canonical["jumlah_baris_dihapus_dedup"] = (
        len(group) - 1
    )
    canonical["jumlah_teks_sebelum_merge"] = int(
        group["_text_count"].sum()
    )
    canonical["jumlah_teks_setelah_merge"] = len(
        merged_segments
    )
    canonical[
        "jumlah_teks_lintas_entitas_dihapus"
    ] = removed_cross_entity

    final_rows.append(canonical)

    if len(group) > 1:
        duplicate_records.append({
            "entity_key": entity_key,
            "title": canonical["title"],
            "jumlah_baris_sumber": len(group),
            "anggota_sumber": canonical["anggota_sumber"],
            "jumlah_teks_setelah_merge": len(
                merged_segments
            ),
        })

df_final = pd.DataFrame(final_rows).reset_index(drop=True)
df_audit_duplikat = pd.DataFrame(duplicate_records)

print("Sebelum dedup:", len(df_raw))
print("Sesudah dedup:", len(df_final))
print("Dihapus      :", len(df_raw) - len(df_final))


In [ ]:

# 9. VALIDASI AKHIR SEBELUM NLP
hard_invalid = (
    df_final["title"].isna()
    | df_final["title"].astype(str).str.strip().eq("")
    | df_final["totalScore"].isna()
    | ~df_final["totalScore"].between(
        1, 5, inclusive="both"
    )
    | df_final["reviewsCount"].isna()
    | (df_final["reviewsCount"] < 1)
    | df_final["text"].isna()
    | df_final["text"].astype(str).str.strip().eq("")
    | (df_final["jumlah_teks_untuk_sentimen"] < 1)
)

df_drop_invalid = df_final[hard_invalid].copy()
df_ready = df_final[~hard_invalid].copy().reset_index(
    drop=True
)

audit_anomaly = df_ready[
    df_ready["street"].isna()
    | df_ready["street"].astype(str).str.strip().eq("")
].copy()

if len(audit_anomaly) > 0:
    audit_anomaly["anomaly_code"] = "MISSING_STREET"
    audit_anomaly["keterangan"] = (
        "Dipertahankan karena telah lolos review entitas; "
        "alamat kosong hanya dicatat sebagai keterbatasan."
    )

print("Hard invalid :", len(df_drop_invalid))
print("Siap NLP     :", len(df_ready))

if len(df_ready) == 0:
    raise ValueError(
        "Output kosong. Periksa parsing totalScore."
    )


In [ ]:

# 10. REKONSILIASI
reconciliation = pd.DataFrame([
    {
        "tahap": "Input hasil review entitas",
        "jumlah_sebelum": len(df_raw),
        "jumlah_dihapus": 0,
        "jumlah_sesudah": len(df_raw),
    },
    {
        "tahap": "Deduplikasi lintas anggota",
        "jumlah_sebelum": len(df_raw),
        "jumlah_dihapus": len(df_raw) - len(df_final),
        "jumlah_sesudah": len(df_final),
    },
    {
        "tahap": "Validasi keras sebelum NLP",
        "jumlah_sebelum": len(df_final),
        "jumlah_dihapus": len(df_drop_invalid),
        "jumlah_sesudah": len(df_ready),
    },
])

summary = pd.DataFrame({
    "indikator": [
        "Baris input",
        "Entity unik sebelum dedup",
        "Duplikat lintas anggota",
        "Baris final siap NLP",
        "Rating minimum",
        "Rating maksimum",
        "Median jumlah ulasan",
        "Jumlah alamat kosong",
    ],
    "nilai": [
        len(df_raw),
        df_raw["entity_key"].nunique(),
        len(df_raw) - len(df_final),
        len(df_ready),
        df_ready["totalScore"].min(),
        df_ready["totalScore"].max(),
        df_ready["reviewsCount"].median(),
        len(audit_anomaly),
    ],
})

display(reconciliation)
display(summary)


In [ ]:

# 11. SELEKSI KOLOM OUTPUT
output_columns = [
    "entity_key",
    "title",
    "totalScore",
    "reviewsCount",
    "street",
    "city",
    "categoryName",
    "url",
    "text",
    "jumlah_teks_untuk_sentimen",
    "anggota_sumber",
    "source_file_list",
    "run_id_list",
    "jumlah_baris_sumber",
    "jumlah_baris_dihapus_dedup",
    "jumlah_teks_sebelum_merge",
    "jumlah_teks_setelah_merge",
    "jumlah_teks_lintas_entitas_dihapus",
]

df_output = df_ready[output_columns].copy()

print("Ukuran output:", df_output.shape)
display(df_output.head(3))


In [ ]:
# 12. EKSPOR SEDERHANA
df_output.to_csv(
    OUTPUT_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

summary.to_csv(
    OUTPUT_AUDIT_CSV,
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

reconciliation.to_csv(
    "02_Hasil_Merger_Audit_Rekonsiliasi.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

df_audit_duplikat.to_csv(
    "02_Hasil_Merger_Audit_Duplikat.csv",
    index=False,
    sep=";",
    encoding="utf-8-sig"
)

print("Berhasil mengespor file hasil merger dan audit ke format CSV.")


In [ ]:

# 13. PEMERIKSAAN WAJIB
assert len(df_raw) == 3464
assert len(df_output) == 3438
assert df_output["entity_key"].nunique() == 3438
assert df_output["totalScore"].between(
    1, 5, inclusive="both"
).all()
assert (df_output["reviewsCount"] >= 1).all()
assert df_output["text"].notna().all()
assert (df_output["jumlah_teks_untuk_sentimen"] >= 1).all()

print("SEMUA VALIDASI LULUS.")
print("Dataset resmi siap NLP:", len(df_output))


In [ ]:
# 14. DOWNLOAD
if RUNNING_IN_COLAB:
    files.download(OUTPUT_CSV)
    files.download(OUTPUT_AUDIT_CSV)
else:
    print("File tersimpan pada folder kerja.")



## File yang dipakai untuk NLP

Gunakan hanya:

`02_Data_Final_Sebelum_NLP_V2.csv`

Jumlah yang diharapkan: **3.438 entitas**.

Kolom audit seperti `anggota_sumber`, `source_file_list`, dan `run_id_list` tetap disimpan untuk dokumentasi, tetapi tidak menjadi fitur sentiment analysis atau K-Means.
